# Cardiovascular Disease Prediction Using Machine Learning

**College Project — Educational Demonstration**

---

## Objective

Develop a machine-learning system that analyses patient clinical data and predicts the likelihood of cardiovascular disease using multiple ML algorithms.

### Models compared:
- Logistic Regression (LR)
- K-Nearest Neighbors (KNN)
- Decision Tree (DT)
- Random Forest (RF)
- Support Vector Machine (SVM)

---

> ⚠️ **Educational Disclaimer:** This notebook is for educational purposes only. Predictions are not medical diagnoses.

## 1. Import Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

from src.config import (
    DATASET_PATH, FIGURES_DIR, MODELS_DIR, REPORTS_DIR, RESULTS_CSV_PATH,
    CANDIDATE_TARGET_COLUMNS, TARGET_COLUMN, DROP_COLUMNS,
    RANDOM_STATE, TEST_SIZE, CV_FOLDS, PRIMARY_METRIC
)
from src.data_preprocessing import (
    load_dataset, detect_target_column, inspect_dataset, full_preprocess
)
from src.eda import (
    generate_all_eda_plots, get_top_correlations,
    plot_target_distribution, plot_correlation_heatmap
)
from src.train_models import (
    split_features, build_preprocessor, train_all_models, save_best_pipeline
)
from src.evaluate_models import (
    evaluate_all_models, select_best_model,
    plot_confusion_matrices, plot_roc_curves, plot_model_comparison, save_results_csv
)
from src.prediction import predict_cardiovascular_disease

sns.set_theme(style='whitegrid', palette='husl')
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

# Create output directories
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print('✅ Libraries imported successfully')

## 2. Load Dataset

In [ ]:
df = load_dataset(DATASET_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Target Column Detection

In [ ]:
TARGET = detect_target_column(df, CANDIDATE_TARGET_COLUMNS, TARGET_COLUMN)
print(f'Target column detected: "{TARGET}"')
print(f'\nClass distribution:')
print(df[TARGET].value_counts())
print(f'\nClass proportions:')
print((df[TARGET].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

## 4. Dataset Inspection

In [ ]:
info = inspect_dataset(df)

print('=== Dataset Dimensions ===')
print(f'Rows: {info["n_rows"]:,}')
print(f'Columns: {info["n_cols"]}')

print('\n=== Column Types ===')
print(f'Numerical  : {info["numeric_cols"]}')
print(f'Categorical: {info["categorical_cols"]}')

print('\n=== Missing Values ===')
if info['missing_df'].empty:
    print('No missing values found ✅')
else:
    display(info['missing_df'])

print(f'\n=== Duplicates ===')
print(f'Duplicate rows: {info["dup_count"]:,} ({info["dup_pct"]}%)')

## 5. Statistical Summary

In [ ]:
df.describe()

## 6. Data Preprocessing

We perform the following:
1. Drop identifier columns
2. Remove duplicate rows
3. Impute missing values (median for numeric, mode for categorical)
4. Remove physiologically invalid blood pressure values
5. Engineer new features (BMI, age in years, pulse pressure)

In [ ]:
df_clean, prep_summary = full_preprocess(df.copy(), DROP_COLUMNS, TARGET)

print('=== Preprocessing Summary ===')
print(f'Dropped identifier columns : {prep_summary["dropped_id_cols"]}')
print(f'Duplicate rows removed     : {prep_summary["duplicates_removed"]}')
print(f'Invalid BP rows removed    : {prep_summary["invalid_bp_removed"]}')
print(f'Engineered features added  : {prep_summary["engineered_features"]}')
print(f'Final dataset shape        : {prep_summary["final_shape"]}')
print(f'Target distribution        : {prep_summary["target_distribution"]}')

## 7. Exploratory Data Analysis

In [ ]:
info2 = inspect_dataset(df_clean)
saved_plots = generate_all_eda_plots(
    df_clean, TARGET, FIGURES_DIR,
    info2['numeric_cols'], info2['categorical_cols']
)
print(f'Generated {len(saved_plots)} plots:')
for p in saved_plots:
    print(f'  {p}')

### 7.1 Display Target Distribution

In [ ]:
fig_path = os.path.join(FIGURES_DIR, '01_target_distribution.png')
if os.path.exists(fig_path):
    from IPython.display import Image
    display(Image(fig_path, width=700))

### 7.2 Numerical Distributions

In [ ]:
fig_path = os.path.join(FIGURES_DIR, '02_numerical_distributions.png')
if os.path.exists(fig_path):
    from IPython.display import Image
    display(Image(fig_path, width=900))

### 7.3 Correlation Heatmap

The heatmap shows pairwise Pearson correlation coefficients. Values close to +1.0 indicate strong positive correlation; values close to -1.0 indicate inverse correlation. Note: **correlation ≠ causation**.

In [ ]:
fig_path = os.path.join(FIGURES_DIR, '05_correlation_heatmap.png')
if os.path.exists(fig_path):
    from IPython.display import Image
    display(Image(fig_path, width=800))

print('\nTop features correlated with target:')
print(get_top_correlations(df_clean, TARGET, top_n=10).to_string())

## 8. Feature Engineering & Train/Test Split

We split features (X) from the target (y), then split into training (80%) and testing (20%) sets.

> **Important:** Preprocessing (scaling, imputation) is fitted ONLY on training data to prevent data leakage.

In [ ]:
X, y, num_cols, cat_cols = split_features(df_clean, TARGET)
print(f'Features: {list(X.columns)}')
print(f'Numeric : {num_cols}')
print(f'Categ.  : {cat_cols}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'\nTrain size: {len(X_train)}')
print(f'Test size : {len(X_test)}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')

## 9. Model Training

Each model is wrapped in a scikit-learn Pipeline:
```
Pipeline → [ColumnTransformer (Imputer + Scaler)] → [Classifier]
```

This ensures no data leakage — transformations are fitted on training data only.

In [ ]:
preprocessor = build_preprocessor(num_cols, cat_cols)

print('Training models...')
pipelines, cv_scores = train_all_models(
    X_train, y_train, preprocessor,
    tune=False, cv_folds=CV_FOLDS, random_state=RANDOM_STATE
)

print('\n=== 5-Fold Cross-Validation ROC-AUC ===')
for model, scores in cv_scores.items():
    print(f'{model:35s}: {scores["mean"]:.4f} ± {scores["std"]:.4f}')

## 10. Model Evaluation

We evaluate all models on the held-out test set (20% of data).

In [ ]:
results_df, raw_results = evaluate_all_models(pipelines, X_test, y_test)

print('=== Test Set Performance ===')
display(results_df[['Accuracy','Precision','Recall','F1 Score','ROC-AUC']].round(4))

### 10.1 Confusion Matrices

Confusion matrices show **True Positives**, **True Negatives**, **False Positives**, and **False Negatives** for each model.

In [ ]:
plot_confusion_matrices(raw_results, y_test, FIGURES_DIR)
fig_path = os.path.join(FIGURES_DIR, '07_confusion_matrices.png')
if os.path.exists(fig_path):
    from IPython.display import Image
    display(Image(fig_path, width=900))

### 10.2 ROC Curves

ROC curves show the tradeoff between True Positive Rate and False Positive Rate. Higher AUC = better model.

In [ ]:
plot_roc_curves(raw_results, y_test, FIGURES_DIR)
fig_path = os.path.join(FIGURES_DIR, '08_roc_curve_comparison.png')
if os.path.exists(fig_path):
    from IPython.display import Image
    display(Image(fig_path, width=700))

### 10.3 Metric Comparison

In [ ]:
plot_model_comparison(results_df, FIGURES_DIR)
fig_path = os.path.join(FIGURES_DIR, '09_model_comparison.png')
if os.path.exists(fig_path):
    from IPython.display import Image
    display(Image(fig_path, width=900))

## 11. Best Model Selection

We select the best model based on **ROC-AUC** as the primary metric.

For a disease-risk classification problem, ROC-AUC is preferred over accuracy because:
- It accounts for class imbalance
- It evaluates the model's ability to distinguish between classes
- Recall (sensitivity) is also important to minimise missed disease cases

In [ ]:
metric_col_map = {'roc_auc': 'ROC-AUC', 'f1': 'F1 Score', 'recall': 'Recall', 'accuracy': 'Accuracy'}
pk = metric_col_map.get(PRIMARY_METRIC, 'ROC-AUC')
best_name = select_best_model(results_df, pk)

print(f'🏆 Best Model: {best_name}')
print(f'   Selection metric: {pk}')
print(f'   Score: {results_df.loc[best_name, pk]:.4f}')
print()
print('All metrics for best model:')
print(results_df.loc[best_name].to_string())

## 12. Save Best Model

In [ ]:
save_best_pipeline(
    pipeline=pipelines[best_name],
    model_name=best_name,
    feature_cols=list(X.columns),
    target_col=TARGET,
    metrics=results_df.loc[best_name].to_dict(),
    dataset_info={'shape': list(df_clean.shape), 'target': TARGET},
    models_dir=MODELS_DIR,
)
save_results_csv(results_df, RESULTS_CSV_PATH)

print(f'✅ Model saved: {MODELS_DIR}/best_model_pipeline.pkl')
print(f'✅ Results saved: {RESULTS_CSV_PATH}')

## 13. Demo Prediction

In [ ]:
# Build a sample patient from the median of each feature
sample_patient = {col: float(df_clean[col].median()) for col in X.columns}

print('Sample patient features:')
for k, v in sample_patient.items():
    print(f'  {k}: {v}')

result = predict_cardiovascular_disease(pipelines[best_name], sample_patient, list(X.columns))

print(f'\n=== Prediction Result ===')
print(f'Predicted Class  : {result["predicted_class"]}')
print(f'Risk Label       : {result["risk_label"]}')
print(f'Confidence       : {result["confidence_pct"]}')
print(f'\n{result["disclaimer"]}')

## 14. Conclusion

### Summary

This notebook demonstrated a complete machine-learning pipeline for cardiovascular disease prediction:

1. **Data Preprocessing** — Handled missing values, duplicates, and invalid entries
2. **Feature Engineering** — Added BMI, age in years, pulse pressure where applicable
3. **EDA** — Visualised distributions, correlations, and class balance
4. **Model Training** — Trained 5 algorithms using proper sklearn Pipelines (no data leakage)
5. **Evaluation** — Compared Accuracy, Precision, Recall, F1-Score, ROC-AUC
6. **Cross-validation** — 5-fold stratified CV for robust evaluation
7. **Best Model** — Selected based on ROC-AUC
8. **Persistence** — Saved entire pipeline for deployment

### Limitations

- Results depend on the quality and representativeness of the dataset
- Model has not been clinically validated
- Real clinical decisions require expert medical evaluation

### ⚠️ Disclaimer

> This project is for educational purposes only. Predictions should never be used as a substitute for professional medical diagnosis.